In [1]:
import pandas as pd

df=pd.read_csv("all_brands.csv")

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   brand        1500 non-null   object
 1   model        1500 non-null   object
 2   trim         258 non-null    object
 3   price        1500 non-null   int64 
 4   year         1500 non-null   int64 
 5   description  1500 non-null   object
 6   place        1500 non-null   object
dtypes: int64(2), object(5)
memory usage: 82.2+ KB


# Preprocessing

In [3]:
# Handling missing values

df.drop(columns=["trim"], inplace=True)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   brand        1500 non-null   object
 1   model        1500 non-null   object
 2   price        1500 non-null   int64 
 3   year         1500 non-null   int64 
 4   description  1500 non-null   object
 5   place        1500 non-null   object
dtypes: int64(2), object(4)
memory usage: 70.4+ KB


In [5]:
# Encoding

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = encoder.fit_transform(df[col])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   brand        1500 non-null   int64
 1   model        1500 non-null   int64
 2   price        1500 non-null   int64
 3   year         1500 non-null   int64
 4   description  1500 non-null   int64
 5   place        1500 non-null   int64
dtypes: int64(6)
memory usage: 70.4 KB


# RandomForestRegressor

In [7]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,  mean_squared_error, r2_score

In [8]:
x=df.drop("price", axis=1)
y=df["price"]

In [9]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [10]:
rf_regressor = RandomForestRegressor(n_estimators=2, random_state=42)

In [11]:
rf_regressor.fit(x_train, y_train)

RandomForestRegressor(n_estimators=2, random_state=42)

In [12]:
y_pred = rf_regressor.predict(x_test)
rf_r2_score=r2_score(y_test, y_pred)
rf_mse=mean_squared_error(y_test, y_pred)


In [13]:
print("Mean squared error: ", rf_mse)
print("R2 score: ", rf_r2_score)

Mean squared error:  32613149.754166666
R2 score:  0.7925502749345089


# RandomForestClassifier

In [41]:
x=df.drop("model", axis=1)
y=df["model"]

In [42]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [43]:
rf_classifier = RandomForestClassifier(n_estimators=2, random_state=42)

In [44]:
rf_classifier.fit(x_train, y_train)

RandomForestClassifier(n_estimators=2, random_state=42)

In [45]:
y_pred = rf_classifier.predict(x_test)
rf_accuracy=accuracy_score(y_test, y_pred)
# rf_confusion_matrix=confusion_matrix(y_test, y_pred)


In [46]:
print("Accuracy score: ", rf_accuracy)

Accuracy score:  0.49666666666666665


# Voting classifier

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier, VotingRegressor


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   brand        1500 non-null   int64
 1   model        1500 non-null   int64
 2   price        1500 non-null   int64
 3   year         1500 non-null   int64
 4   description  1500 non-null   int64
 5   place        1500 non-null   int64
dtypes: int64(6)
memory usage: 70.4 KB


In [16]:
x=df.drop("model", axis=1)
y=df["model"]

In [17]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [18]:
lr=LogisticRegression(max_iter=1000)
dt=DecisionTreeClassifier()
svc=SVC(probability=True)

# Hard Voting

In [19]:
hard_voting=VotingClassifier(
    estimators=[("lr", lr), ("dt", dt), ("svc", svc)],
    voting="hard"
)

In [20]:
hard_voting.fit(x_train, y_train)

/Users/asadbek/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


VotingClassifier(estimators=[('lr', LogisticRegression(max_iter=1000)),
                             ('dt', DecisionTreeClassifier()),
                             ('svc', SVC(probability=True))])

In [47]:
y_pred_hard = hard_voting.predict(x_test)
hard_accuracy=accuracy_score(y_test, y_pred_hard)

In [48]:
print("Hard voting accuracy: ", hard_accuracy)

Hard voting accuracy:  0.3933333333333333


# Soft voting

In [23]:
soft_voting=VotingClassifier(
    estimators=[("lr", lr), ("dt", dt), ("svc", svc)],
    voting="soft"
)

In [24]:
soft_voting.fit(x_train, y_train)

/Users/asadbek/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


VotingClassifier(estimators=[('lr', LogisticRegression(max_iter=1000)),
                             ('dt', DecisionTreeClassifier()),
                             ('svc', SVC(probability=True))],
                 voting='soft')

In [50]:
y_pred_soft = soft_voting.predict(x_test)
soft_accuracy=accuracy_score(y_test, y_pred_soft)

In [51]:
print("Soft voting accuracy: ", soft_accuracy)
print("Hard voting accuracy: ", hard_accuracy)


Soft voting accuracy:  0.55
Hard voting accuracy:  0.3933333333333333


# Voting Regressor

In [27]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

In [28]:
x=df.drop('price',axis=1)
y=df['price']

In [29]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [30]:
lr = LinearRegression()
ridge = Ridge()
dt = DecisionTreeRegressor()

In [31]:
# Voting Regressor
voting_reg = VotingRegressor(
    estimators=[('lr', lr), ('ridge', ridge), ('dt', dt)]
)

In [32]:
voting_reg.fit(x_train, y_train)


VotingRegressor(estimators=[('lr', LinearRegression()), ('ridge', Ridge()),
                            ('dt', DecisionTreeRegressor())])

In [36]:
y_pred = voting_reg.predict(x_test)
voting_mse = mean_squared_error(y_test, y_pred)
voting_r2 = r2_score(y_test, y_pred)


In [37]:
print("Voting Regressor MSE:", voting_mse)


Voting Regressor MSE: 39571896.59281972


In [38]:
print("Voting Regressor R2:", voting_r2)


Voting Regressor R2: 0.7482862241034633


# Regression

In [39]:
from tabulate import tabulate

results = [
    ["Random Forest Regressor", rf_mse, rf_r2_score],
    ["Voting Regressor", voting_mse, voting_r2]
]   

tabulated_results = tabulate(results, headers=["Model", "MSE", "R2 Score"], tablefmt="grid")

In [40]:
print(tabulated_results)

+-------------------------+-------------+------------+
| Model                   |         MSE |   R2 Score |
+=========================+=============+============+
| Random Forest Regressor | 3.26131e+07 |   0.79255  |
+-------------------------+-------------+------------+
| Voting Regressor        | 3.95719e+07 |   0.748286 |
+-------------------------+-------------+------------+


# Classification

In [52]:
results = [
    ["Random Forest Classifier", rf_accuracy],
    ["Hard Voting Classifier", hard_accuracy],
    ["Soft Voting Classifier", soft_accuracy]
]   

tabulated_results = tabulate(results, headers=["Model", "Accuracy"], tablefmt="grid")

In [53]:
print(tabulated_results)

+--------------------------+------------+
| Model                    |   Accuracy |
+==========================+============+
| Random Forest Classifier |   0.496667 |
+--------------------------+------------+
| Hard Voting Classifier   |   0.393333 |
+--------------------------+------------+
| Soft Voting Classifier   |   0.55     |
+--------------------------+------------+
